In [1]:
!pip install -q langchain-groq langgraph

In [16]:
!pip install -q "unstructured[pdf]"
!apt install poppler-utils tesseract-ocr

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
poppler-utils is already the newest version (22.02.0-2ubuntu0.10).
0 upgraded, 0 newly installed, 0 to remove and 38 not upgraded.


In [15]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from google.colab import userdata
import os
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
llm = ChatGroq(model="openai/gpt-oss-120b")

In [37]:
import fitz
import re

In [27]:
def find_toc_pages(doc, search_limit=20):
    toc_pages = []
    for i in range(min(len(doc), search_limit)):
        page = doc[i]
        text = page.get_text("text", sort=True).lower()
        if "contents" in text or "table of contents" in text:
            toc_pages.append(i)
    return toc_pages

def toc_raw_to_hierarchy(toc_raw):
    messages = [SystemMessage("""
        You are a document hierarchy generator. Your task is to generate a python dictionary.
        You are given the raw text on the table of contents page of a document.
        analyze it properly and structure it into a neat json object.
        The document hierarchy is only needed till depth level 2. the schema of the json object is as follows
        {
            "section_name": str
            "section_children": list[str]
        }
        You must follow this schema.
        YOur output should be json object and it will be evaluated as it is so do not give boilerplate text or backticks
        """)]
    messages.append(HumanMessage("here is the raw text on the table of contents page of the document"))
    messages.append(HumanMessage(toc_raw))

    res = llm.with_structured_output(method="json_mode").invoke(messages)
    return res

def toc_to_hierarchy(toc):
    hierarchy = []
    for i in range(len(toc)):
        if toc[i][0] == 1:
            new_section = {
                "section_name": toc[i][1],
                "section_children": []
            }
            new_section_children = []
            for j in range(i+1, len(toc)):
                if toc[j][0] == 2:
                    new_section_children.append(toc[j][1])
                elif toc[j][0] == 1:
                    break
            new_section["section_children"] = new_section_children
            hierarchy.append(new_section)
    return hierarchy

def get_doc_hierarchy(pdf_path):
    doc = fitz.open(pdf_path)
    toc = doc.get_toc()
    if toc:
        return toc_to_hierarchy(toc)

    toc_pages = find_toc_pages(doc)

    if toc_pages:
        toc_raw = ""
        for i in toc_pages:
            toc_raw += doc[i].get_text("text")
        return toc_raw_to_hierarchy(toc_raw)

    return None


In [17]:
from unstructured.partition.pdf import partition_pdf
elements = partition_pdf(
    filename="report.pdf",
    strategy="fast"
    )

In [28]:
doc_hierarchy = get_doc_hierarchy("report.pdf")

In [32]:
section_map = {sec["section_name"]: set(sec.get("section_children", []))
                for sec in doc_hierarchy}

# All known section titles (for detection)
all_sections = set(section_map.keys())
for children in section_map.values():
    all_sections.update(children)

['Introduction',
 'Background',
 'Model Architecture',
 'Why Self-Attention',
 'Training',
 'Results',
 'Conclusion']

In [46]:
def get_chunks_from_hierarchy(doc_hierarchy, elements):
    all_sections = [section["section_name"] for section in doc_hierarchy]

    # Normalize titles for matching
    def normalize(s: str) -> str:
        return re.sub(r"\s+", " ", s.strip()).lower()

    normalized_sections = {normalize(s): s for s in all_sections}

    chunks = {sec: [] for sec in all_sections}  # all sections present
    current_section = None

    for el in elements:
        text = el.text
        if not text:
            continue

        # Detect if element starts a new section
        norm_text = normalize(text)
        if norm_text in normalized_sections:
            current_section = normalized_sections[norm_text]
            # Optionally we can store the section heading itself as an element
            chunks[current_section].append(el)
            continue

        # If we are inside a section, add element to it
        if current_section:
            chunks[current_section].append(el)
        else:
            # Belongs to no recognized section
            chunks.setdefault("UNASSIGNED", []).append(el)